In [1]:
import sys
sys.path.append("../ingestion/python/src")
sys.path.append("../ingestion/python/LangGraph_Agent")

from utils import *
from langgraph.graph import StateGraph, START, END
from IPython.display import Image, display


from APIendpoint import PlacesAPI
from database import Database
import pandas as pd
import os
from dotenv import load_dotenv
load_dotenv(override=True)

db = Database()

c:\APPS\Python312\Lib\site-packages\langgraph\cache\base\__init__.py:8: LangChainPendingDeprecationWarning: The default value of `allowed_objects` will change in a future version. Pass an explicit value (e.g., allowed_objects='messages' or allowed_objects='core') to suppress this warning.
  from langgraph.checkpoint.serde.jsonplus import JsonPlusSerializer


In [3]:
# ============================================================
# Comparaison Bronze / Staging / Silver — détection des retards de pipeline
# ============================================================

# 1. Volumes bruts à chaque étape
counts = db.execute("""
    SELECT
        (SELECT COUNT(*) FROM raw.job_offer) AS bronze_total,
        (SELECT COUNT(*) FROM staging.enriched_offers) AS staging_total,
        (SELECT COUNT(*) FROM analytics.job_offer) AS silver_total;
""")

# 2. Offres présentes en Bronze mais jamais arrivées en Staging (enrichissement pas encore lancé / échoué)
missing_in_staging = db.execute("""
    SELECT COUNT(*) AS missing_in_staging
    FROM raw.job_offer b
    LEFT JOIN staging.enriched_offers s ON s.id_offer = b.id_job
    WHERE s.id_offer IS NULL;
""")

# Détail par source, pour voir si le retard est concentré sur une API précise
missing_in_staging_by_source = db.execute("""
    SELECT b.api_source, COUNT(*) AS n
    FROM raw.job_offer b
    LEFT JOIN staging.enriched_offers s ON s.id_offer = b.id_job
    WHERE s.id_offer IS NULL
    GROUP BY b.api_source
    ORDER BY n DESC;
""")

# 3. Offres présentes en Staging mais jamais poussées en Silver (transformation pas encore lancée / échouée)
missing_in_silver = db.execute("""
    SELECT COUNT(*) AS missing_in_silver
    FROM staging.enriched_offers s
    LEFT JOIN analytics.job_offer jo ON jo.id_offer = s.id_offer
    WHERE jo.id_offer IS NULL;
""")

# Detail : ces offres ont-elles un company_name exploitable, ou sont-elles rejetées (donc normal qu'elles ne soient pas en Silver) ?
missing_in_silver_reason = db.execute("""
    SELECT
        CASE
            WHEN s.raw_result->>'company_name' IS NULL
              OR s.raw_result->>'company_name' IN ('', 'null', 'Empresa confidencial')
            THEN 'company invalide (rejet normal)'
            ELSE 'company valide mais absent de Silver (retard ou bug)'
        END AS reason,
        COUNT(*) AS n
    FROM staging.enriched_offers s
    LEFT JOIN analytics.job_offer jo ON jo.id_offer = s.id_offer
    WHERE jo.id_offer IS NULL
    GROUP BY reason;
""")

# 4. Âge du retard — depuis combien de temps ces offres bloquées attendent
staging_lag_stats = db.execute("""
    SELECT
        MIN(b.collected_at) AS oldest_unenriched,
        MAX(b.collected_at) AS newest_unenriched,
        NOW() - MIN(b.collected_at) AS max_lag
    FROM raw.job_offer b
    LEFT JOIN staging.enriched_offers s ON s.id_offer = b.id_job
    WHERE s.id_offer IS NULL;
""")

silver_lag_stats = db.execute("""
    SELECT
        MIN(s.collected_at) AS oldest_unpushed,
        MAX(s.collected_at) AS newest_unpushed,
        NOW() - MIN(s.collected_at) AS max_lag
    FROM staging.enriched_offers s
    LEFT JOIN analytics.job_offer jo ON jo.id_offer = s.id_offer
    WHERE jo.id_offer IS NULL;
""")

# 5. Company null en Bronze — combien résolus par le LLM, combien encore null
bronze_company_null = db.execute("""
    SELECT COUNT(*) AS bronze_company_null
    FROM raw.job_offer
    WHERE company IS NULL OR company = '';
""")

resolved_by_llm = db.execute("""
    SELECT COUNT(*) AS resolved_by_llm
    FROM raw.job_offer b
    JOIN staging.enriched_offers s ON s.id_offer = b.id_job
    WHERE (b.company IS NULL OR b.company = '')
      AND s.raw_result->>'company_name' IS NOT NULL
      AND s.raw_result->>'company_name' NOT IN ('', 'null', 'Empresa confidencial');
""")

still_null_after_llm = db.execute("""
    SELECT COUNT(*) AS still_null_after_llm
    FROM raw.job_offer b
    JOIN staging.enriched_offers s ON s.id_offer = b.id_job
    WHERE (b.company IS NULL OR b.company = '')
      AND (s.raw_result->>'company_name' IS NULL
           OR s.raw_result->>'company_name' IN ('', 'null', 'Empresa confidencial'));
""")

llm_resolved_details = db.execute("""
    SELECT b.api_source, COUNT(*) AS n
    FROM raw.job_offer b
    JOIN staging.enriched_offers s ON s.id_offer = b.id_job
    WHERE (b.company IS NULL OR b.company = '')
      AND (s.raw_result->>'company_name' IS NULL
           OR s.raw_result->>'company_name' IN ('', 'null', 'Empresa confidencial'))
    GROUP BY b.api_source;
""")

# ============================================================
# Regroupement et affichage
# ============================================================
r = {
    "counts_by_layer": counts,
    "missing_in_staging": missing_in_staging,
    "missing_in_staging_by_source": missing_in_staging_by_source,
    "missing_in_silver": missing_in_silver,
    "missing_in_silver_reason": missing_in_silver_reason,
    "staging_lag_stats": staging_lag_stats,
    "silver_lag_stats": silver_lag_stats,
    "bronze_company_null": bronze_company_null,
    "resolved_by_llm": resolved_by_llm,
    "still_null_after_llm": still_null_after_llm,
    "llm_resolved_details": llm_resolved_details,
}

for key, value in r.items():
    print(f"\n{'='*60}\n{key}\n{'='*60}")
    print(value)


counts_by_layer
[{'bronze_total': 2415, 'staging_total': 2415, 'silver_total': 1759}]

missing_in_staging
[{'missing_in_staging': 0}]

missing_in_staging_by_source
[]

missing_in_silver
[{'missing_in_silver': 656}]

missing_in_silver_reason
[{'reason': 'company valide mais absent de Silver (retard ou bug)', 'n': 489}, {'reason': 'company invalide (rejet normal)', 'n': 167}]

staging_lag_stats
[{'oldest_unenriched': None, 'newest_unenriched': None, 'max_lag': None}]

silver_lag_stats
[{'oldest_unpushed': datetime.datetime(2026, 6, 21, 21, 59, 18, 605322, tzinfo=datetime.timezone.utc), 'newest_unpushed': datetime.datetime(2026, 7, 21, 19, 8, 26, 444406, tzinfo=datetime.timezone.utc), 'max_lag': datetime.timedelta(days=30, seconds=5567, microseconds=857345)}]

bronze_company_null
[{'bronze_company_null': 205}]

resolved_by_llm
[{'resolved_by_llm': 38}]

still_null_after_llm
[{'still_null_after_llm': 167}]

llm_resolved_details
[{'api_source': 'careerjet', 'n': 167}]


In [2]:
# 1. Exécution des requêtes individuelles
bronze_company_null = db.execute("""
    SELECT COUNT(*) AS bronze_company_null
    FROM raw.job_offer
    WHERE company IS NULL OR company = '';
""")

resolved_by_llm = db.execute("""
    SELECT COUNT(*) AS resolved_by_llm
    FROM raw.job_offer b
    JOIN staging.enriched_offers s ON s.id_offer = b.id_job
    WHERE (b.company IS NULL OR b.company = '')
      AND s.raw_result->>'company_name' IS NOT NULL
      AND s.raw_result->>'company_name' NOT IN ('', 'null', 'Empresa confidencial');
""")

still_null_after_llm = db.execute("""
    SELECT COUNT(*) AS still_null_after_llm
    FROM raw.job_offer b
    JOIN staging.enriched_offers s ON s.id_offer = b.id_job
    WHERE (b.company IS NULL OR b.company = '')
      AND (s.raw_result->>'company_name' IS NULL
           OR s.raw_result->>'company_name' IN ('', 'null', 'Empresa confidencial'));
""")

# Ces 149 sont-elles concentrées sur une source (careerjet vs jsearch) ?
llm_resolved_details = db.execute("""
SELECT
    b.api_source,
    COUNT(*) AS n
FROM raw.job_offer b
JOIN staging.enriched_offers s ON s.id_offer = b.id_job
WHERE (b.company IS NULL OR b.company = '')
  AND (s.raw_result->>'company_name' IS NULL
       OR s.raw_result->>'company_name' IN ('', 'null', 'Empresa confidencial'))
GROUP BY b.api_source;
""")

# 2. Regroupement et affichage de tous les résultats
r = {
    "bronze_company_null": bronze_company_null,
    "resolved_by_llm": resolved_by_llm,
    "still_null_after_llm": still_null_after_llm,
    "llm_resolved_details": llm_resolved_details
}

print(r)


{'bronze_company_null': [{'bronze_company_null': 205}], 'resolved_by_llm': [{'resolved_by_llm': 38}], 'still_null_after_llm': [{'still_null_after_llm': 167}], 'llm_resolved_details': [{'api_source': 'careerjet', 'n': 167}]}


In [3]:
q = """SELECT
    b.id_job,
    LENGTH(b.offer_description) AS description_length,
    b.collected_at
FROM raw.job_offer b
JOIN staging.enriched_offers s ON s.id_offer = b.id_job
WHERE b.api_source = 'careerjet'
  AND (b.company IS NULL OR b.company = '')
  AND (s.raw_result->>'company_name' IS NULL
       OR s.raw_result->>'company_name' IN ('', 'null', 'Empresa confidencial'))
ORDER BY b.collected_at
LIMIT 20;"""

r = db.execute(q)
r

[{'id_job': 'cj_c54e2eeee7d8ae0c7ae109464f711008',
  'description_length': 263,
  'collected_at': datetime.datetime(2026, 6, 9, 22, 45, 48, 488360, tzinfo=datetime.timezone.utc)},
 {'id_job': 'cj_59cf47c22d3120c4c544a4cabfc915ca',
  'description_length': 269,
  'collected_at': datetime.datetime(2026, 6, 9, 22, 45, 48, 488475, tzinfo=datetime.timezone.utc)},
 {'id_job': 'cj_fef20ce3a0164e2b99fa0c3c3fdd577a',
  'description_length': 262,
  'collected_at': datetime.datetime(2026, 6, 9, 22, 45, 48, 488516, tzinfo=datetime.timezone.utc)},
 {'id_job': 'cj_19971103e452c44dd99b69afe7026187',
  'description_length': 306,
  'collected_at': datetime.datetime(2026, 6, 9, 22, 45, 48, 488608, tzinfo=datetime.timezone.utc)},
 {'id_job': 'cj_fdd47bba11be1f542a3df663db7324ee',
  'description_length': 260,
  'collected_at': datetime.datetime(2026, 6, 9, 22, 45, 48, 488628, tzinfo=datetime.timezone.utc)},
 {'id_job': 'cj_48afdfe1187458d5f203eda5f2c2609f',
  'description_length': 272,
  'collected_at': da